# Personnes traitées pour hypertension artérielle : profil, recours aux soins et coûts

**Master en Sciences Biomédicales, Université de Genève**
Module d'économie de la santé. Travaux pratiques : analyse descriptive de données de remboursement de l'assurance obligatoire des soins.

## Question de recherche

Quel est le profil des personnes traitées pour hypertension artérielle parmi les assuré·es d'un assureur-maladie, comment recourent-elles au système de soins, et quels coûts représentent-elles pour l'assurance obligatoire des soins en comparaison des autres assuré·es ?

Avant d'aborder cette question, les travaux décrivent la population assurée et la distribution des dépenses : qui coûte, pour quelles raisons, et avec quelle concentration.

## Objectifs

À l'issue de ces travaux pratiques, les étudiant·es seront capables de décrire les principaux déterminants des dépenses de santé (âge, sexe, cause des prestations) et leur concentration, d'interpréter une analyse descriptive simple de données de remboursement, d'en identifier les principales limites, et d'en tirer des éléments d'interprétation à l'intention d'un·e décideur·e de santé publique.

## Données

Les données correspondent à une extraction des décomptes de l'année 2024 d'un assureur-maladie de Suisse romande, portant sur 15 000 assuré·es adultes. Chaque prestation indique sa cause (maladie, accident ou maternité) et le prestataire qui la facture. Les données sont **entièrement synthétiques** : produites par simulation afin de reproduire les caractéristiques des données réelles, elles ne contiennent aucune information relative à des personnes existantes. La description des variables figure dans le fichier `dictionnaire_donnees.md` ; le fichier des médicaments ne contient qu'une sélection de substances.

## Travail attendu

Chaque groupe rédige une note d'une page destinée à un·e responsable de santé publique, présentant la population concernée, les traitements et le recours aux soins, les coûts associés, deux ou trois messages principaux et une limite d'interprétation.

## Organisation du document

Les cellules s'exécutent dans l'ordre (Maj + Entrée). Le code est fourni ; sa compréhension détaillée n'est pas requise, les commentaires (lignes précédées de `#`) en indiquent l'objet. Chaque section se termine par des questions destinées à la discussion en groupe, dont les réponses alimentent la note. Les paramètres pouvant être modifiés sont signalés comme tels ; les analyses complémentaires sont facultatives.

| Section | Contenu | Durée indicative |
|---|---|---|
| 1 | Prise en main et qualité des données | 20 min |
| 2 | Qui sont les assuré·es ? | 15 min |
| 3 | Qui coûte, et pourquoi ? | 45 min |
| 4 | Personnes traitées pour hypertension : identification, profil, traitements, recours aux soins, coûts | 80 min |
| 5 | Synthèse et rédaction de la note | 20 min |

In [ ]:
# @title Préparation de l'environnement (à exécuter sans modification)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", context="notebook", font_scale=0.95)
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.titleweight"] = "bold"
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.0f}".replace(",", "'") if abs(x) >= 100 else f"{x:.1f}")

ANNEE = 2024
GROUPES = ["HTA traitée", "Autres assuré·es"]
COULEURS = {"HTA traitée": "#C0392B", "Autres assuré·es": "#7F8C8D"}          # un groupe mis en évidence, le reste en gris
COULEURS_SEXE = {"F": "#EB6834", "M": "#2A78D6"}
COULEURS_CAUSE = {"maladie": "#2A78D6", "maternite": "#EB6834", "accident": "#1BAF7A"}
CATEGORIELLES = ["#2A78D6", "#EB6834", "#1BAF7A", "#EDA100", "#E87BA4", "#008300", "#6250D6", "#E34948"]
RAMPE = ["#86B6EF", "#3987E5", "#1C5CAB", "#0D366B"]                          # catégories ordonnées, du clair au foncé
CLASSES_OFSP = [19, 26, 36, 46, 56, 66, 76, 86, 120]                         # classes d'âge de la statistique fédérale
LIBELLES_OFSP = ["19-25", "26-35", "36-45", "46-55", "56-65", "66-75", "76-85", "86+"]


def chf(x):
    """Montant en CHF avec séparateur de milliers."""
    return f"{x:,.0f}".replace(",", "'")


def axe_pct(ax, axe="y"):
    """Affiche un axe en pourcentage."""
    (ax.yaxis if axe == "y" else ax.xaxis).set_major_formatter(mtick.PercentFormatter(1))


def axe_chf(ax, axe="y"):
    """Affiche un axe en CHF avec séparateur de milliers."""
    formateur = mtick.FuncFormatter(lambda v, _: f"{v:,.0f}".replace(",", "'"))
    (ax.yaxis if axe == "y" else ax.xaxis).set_major_formatter(formateur)


print("Environnement prêt.")

In [ ]:
# @title Chargement des données
# Enseignant·e : adresse du dossier contenant les fichiers CSV (ex. dépôt GitHub, lien « raw »)
URL_DONNEES = "https://raw.githubusercontent.com/daderidd/heco-tp-hypertension/main/donnees/"


def lire(nom, **options):
    local = Path("donnees") / nom                       # dossier « donnees » déposé dans Colab
    source = local if local.exists() else URL_DONNEES + nom
    return pd.read_csv(source, **options)


assures = lire("assures.csv", parse_dates=["debut_couverture", "fin_couverture"])
prestations = lire("prestations.csv", parse_dates=["date_prestation", "date_decompte"])
medicaments = lire("medicaments.csv", parse_dates=["date_delivrance"])
referentiel = lire("referentiel_atc.csv")
prestataires = lire("referentiel_prestataires.csv")

print(f"assures      : {len(assures):>8,} lignes")
print(f"prestations  : {len(prestations):>8,} lignes")
print(f"medicaments  : {len(medicaments):>8,} lignes")

---
## 1. Prise en main et qualité des données

Les données comprennent trois tables principales, reliées par l'identifiant `id_assure`, et deux référentiels :

```
assures                     1 ligne par personne assurée
   ├── prestations          1 ligne par position de facture (cause, type de prestation, prestataire)
   └── medicaments          1 ligne par médicament délivré (code ATC, prestataire)
referentiel_atc             code ATC → substance et groupe thérapeutique
referentiel_prestataires    id_prestataire → catégorie, canton, spécialité, statut juridique
```

In [ ]:
# Les 5 premières lignes de chaque table
display(assures.head())
display(prestations.head())
display(medicaments.head())

Les données administratives comportent fréquemment des erreurs et des incohérences liées aux processus de facturation. Les contrôles suivants précèdent toute analyse.

In [ ]:
assures["age"] = ANNEE - assures["annee_naissance"]          # âge au 31 décembre
colonnes_prest = [c for c in prestations.columns if c != "id_ligne"]   # tout sauf l'identifiant de ligne
colonnes_med = [c for c in medicaments.columns if c != "id_ligne"]

controles = pd.Series({
    "Âge > 105 ans": (assures["age"] > 105).sum(),
    "Sexe manquant": assures["sexe"].isna().sum(),
    "Canton manquant": assures["canton"].isna().sum(),
    "Prestations : lignes en double": prestations.duplicated(subset=colonnes_prest).sum(),
    "Prestations : montant négatif": (prestations["cout_chf"] < 0).sum(),
    "Prestations : montant nul": (prestations["cout_chf"] == 0).sum(),
    f"Prestations : date hors {ANNEE}": (prestations["date_prestation"].dt.year != ANNEE).sum(),
    "Médicaments : lignes en double": medicaments.duplicated(subset=colonnes_med).sum(),
    "Médicaments : code ATC mal écrit": (medicaments["code_atc"] != medicaments["code_atc"].str.strip().str.upper()).sum(),
}, name="Nombre")
controles.to_frame()

In [ ]:
# Exemple : lignes d'une même personne, même date et même type de prestation qu'une ligne à montant négatif
cas = prestations[prestations["cout_chf"] < 0].iloc[0]
prestations[(prestations["id_assure"] == cas["id_assure"])
            & (prestations["date_prestation"] == cas["date_prestation"])
            & (prestations["type_prestation"] == cas["type_prestation"])]

**Règles de préparation retenues.** Les tables d'origine ne sont pas modifiées ; les analyses portent sur des copies. Les âges supérieurs à 105 ans sont considérés comme manquants, les doublons exacts sont supprimés, les prestations antérieures à 2024 sont exclues et les codes ATC sont normalisés. Les lignes d'annulation sont conservées dans le calcul des montants, puisqu'elles compensent la facture d'origine. Pour dénombrer les recours, les lignes d'une même personne, à une même date, pour un même type de prestation et une même cause, sont regroupées en un **contact** lorsque leur montant net est positif.

In [ ]:
# Assuré·es
assures_ok = assures.copy()
assures_ok.loc[assures_ok["age"] > 105, "age"] = np.nan
assures_ok["classe_age"] = pd.cut(assures_ok["age"], bins=[18, 39, 49, 59, 69, 79, 110],
                                  labels=["19-39", "40-49", "50-59", "60-69", "70-79", "80+"])
assures_ok["classe_ofsp"] = pd.cut(assures_ok["age"], bins=CLASSES_OFSP, right=False, labels=LIBELLES_OFSP)

# Médicaments
med_ok = medicaments.drop_duplicates(subset=colonnes_med).copy()
med_ok["code_atc"] = med_ok["code_atc"].str.strip().str.upper()

# Prestations
prest_ok = prestations.drop_duplicates(subset=colonnes_prest)
prest_ok = prest_ok[prest_ok["date_prestation"].dt.year == ANNEE].copy()

# Contacts : une personne, un jour, un type de prestation, une cause, montant net positif
contacts = (prest_ok
            .groupby(["id_assure", "date_prestation", "type_prestation", "cause"], as_index=False)
            .agg(cout_chf=("cout_chf", "sum"), specialite=("specialite", "first")))
contacts = contacts[contacts["cout_chf"] > 0]

print(f"Prestations : {len(prestations):,} lignes → {len(prest_ok):,} lignes nettoyées → {len(contacts):,} contacts")
print(f"Médicaments : {len(medicaments):,} lignes → {len(med_ok):,} délivrances")

**Questions**
1. Quelle est l'unité d'observation de chacune des trois tables principales ?
2. Quelle information, nécessaire à l'étude de l'hypertension, ces données ne contiennent-elles pas ?
3. Comment interpréter la succession de lignes de l'exemple ci-dessus ? Quelles conséquences aurait un dénombrement des lignes pour mesurer le nombre de consultations ?

**Réponses du groupe** *(double-cliquer sur la cellule pour la modifier)*

---
## 2. Qui sont les assuré·es ?

La structure par âge et sexe de la population et ses choix d'assurance influencent fortement les dépenses observées. Ils sont décrits avant toute comparaison.

In [ ]:
# Pyramide des âges (classes de 5 ans ; la première classe compte les 19 à 24 ans)
bornes = [19] + list(range(25, 100, 5)) + [120]
noms = [f"{a}-{b - 1}" for a, b in zip(bornes[:-2], bornes[1:-1])] + ["95+"]
pyramide = (assures_ok.dropna(subset=["age", "sexe"])
            .assign(classe=lambda d: pd.cut(d["age"], bornes, right=False, labels=noms))
            .groupby(["classe", "sexe"], observed=False).size().unstack(fill_value=0))

fig, ax = plt.subplots(figsize=(7, 5))
y = np.arange(len(pyramide))
ax.barh(y, -pyramide["M"], height=0.85, color=COULEURS_SEXE["M"], label="Hommes")
ax.barh(y, pyramide["F"], height=0.85, color=COULEURS_SEXE["F"], label="Femmes")
ax.set_yticks(y, pyramide.index)
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{abs(v):,.0f}".replace(",", "'")))
ax.axvline(0, color="black", lw=0.8)
ax.legend(loc="upper right")
ax.set(title="Pyramide des âges des assuré·es", xlabel="Nombre de personnes", ylabel="Âge (ans)");

In [ ]:
# Franchise choisie selon l'âge (franchises intermédiaires regroupées)
GROUPES_FRANCHISE = {300: "300", 500: "500", 1000: "1000-2000", 1500: "1000-2000", 2000: "1000-2000", 2500: "2500"}
franchise = pd.crosstab(assures_ok["classe_ofsp"], assures_ok["franchise_chf"].map(GROUPES_FRANCHISE),
                        normalize="index")[["300", "500", "1000-2000", "2500"]]
ax = franchise.plot.barh(stacked=True, color=RAMPE, width=0.8, edgecolor="white", linewidth=1, figsize=(8, 4))
ax.invert_yaxis()
axe_pct(ax, "x")
ax.legend(title="Franchise (CHF)", bbox_to_anchor=(1, 1), loc="upper left")
ax.set(title="Franchise choisie selon l'âge", xlabel="Part des assuré·es", ylabel="Classe d'âge");

In [ ]:
# Modèle d'assurance selon le canton
MODELES = ["standard", "medecin_de_famille", "hmo", "telmed"]
modeles = pd.crosstab(assures_ok["canton"], assures_ok["modele_assurance"], normalize="index")[MODELES]
ax = modeles.plot.barh(stacked=True, color=CATEGORIELLES[:4], width=0.8, edgecolor="white", linewidth=1, figsize=(8, 3.5))
ax.invert_yaxis()
axe_pct(ax, "x")
ax.legend(title="Modèle", bbox_to_anchor=(1, 1), loc="upper left")
ax.set(title="Modèle d'assurance selon le canton", xlabel="Part des assuré·es", ylabel="");

**Questions**
1. Comment les assuré·es se répartissent-ils selon l'âge et le sexe ? Que se passe-t-il au-delà de 80 ans ?
2. Comment le choix de la franchise évolue-t-il avec l'âge ? Pourquoi les personnes âgées choisissent-elles presque toutes la franchise minimale ? Quelle particularité présentent les 19 à 25 ans ?
3. Les modèles d'assurance diffèrent-ils d'un canton à l'autre ? Quelle hypothèse pour le Valais ?

**Réponses du groupe** *(double-cliquer sur la cellule pour la modifier)*

---
## 3. Qui coûte, et pourquoi ?

Le coût est défini comme le montant décompté à la charge de l'assurance obligatoire des soins en 2024, prestations et médicaments confondus. Pour les séjours hospitaliers, seule la part financée par l'assureur (45 %) figure dans les données ; la part cantonale (55 %) n'y apparaît pas. Les médicaments, dont la cause n'est pas indiquée, sont rattachés à la maladie.

In [ ]:
# Coût par personne et par cause
par_cause = (prest_ok.pivot_table(index="id_assure", columns="cause", values="cout_chf", aggfunc="sum")
             .reindex(columns=["maladie", "maternite", "accident"]))
couts_cause = par_cause.reindex(assures_ok["id_assure"]).fillna(0)
couts_cause["maladie"] += med_ok.groupby("id_assure")["cout_chf"].sum().reindex(couts_cause.index).fillna(0)
couts_cause["total"] = couts_cause[["maladie", "maternite", "accident"]].sum(axis=1)
couts_cause = couts_cause.join(assures_ok.set_index("id_assure")[["sexe", "age", "classe_ofsp", "couverture_accident"]])

(couts_cause[["maladie", "maternite", "accident", "total"]].sum() / couts_cause["total"].sum()).map("{:.1%}".format).to_frame("Part des coûts")

### 3.1 Dépenses selon l'âge et le sexe

In [ ]:
moyennes_sexe = couts_cause.groupby(["classe_ofsp", "sexe"], observed=True)["total"].mean().unstack()
fig, ax = plt.subplots()
for sexe, nom in [("F", "Femmes"), ("M", "Hommes")]:
    ax.plot(moyennes_sexe.index.astype(str), moyennes_sexe[sexe], marker="o", ms=6, lw=2, color=COULEURS_SEXE[sexe], label=nom)
axe_chf(ax)
ax.legend()
ax.set(title="Coût moyen par personne selon l'âge et le sexe", xlabel="Classe d'âge", ylabel="CHF");

### 3.2 Pourquoi les femmes de 20 à 45 ans coûtent-elles davantage ?

La variable `cause` distingue les prestations liées à la **maladie**, à la **maternité** (suivi de grossesse, accouchement, suites de couches) et aux **accidents**.

In [ ]:
par_cause_age = couts_cause.groupby(["sexe", "classe_ofsp"], observed=True)[["maladie", "maternite", "accident"]].mean()
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, (sexe, nom) in zip(axes, [("F", "Femmes"), ("M", "Hommes")]):
    par_cause_age.loc[sexe].plot.bar(stacked=True, ax=ax, width=0.8, edgecolor="white", linewidth=1, legend=False,
                                     color=[COULEURS_CAUSE[c] for c in ["maladie", "maternite", "accident"]])
    ax.set(title=nom, xlabel="Classe d'âge", ylabel="CHF par personne")
    ax.tick_params(axis="x", rotation=0)
axe_chf(axes[0])
axes[1].legend(["Maladie", "Maternité", "Accident"], bbox_to_anchor=(1, 1), loc="upper left")
fig.suptitle("Coût moyen par personne selon la cause", fontweight="bold");

In [ ]:
# Écart entre femmes et hommes de 26 à 35 ans, et part expliquée par la maternité
jeunes = couts_cause[couts_cause["age"].between(26, 35)].groupby("sexe")[["maladie", "maternite", "accident", "total"]].mean()
ecart = jeunes.loc["F", "total"] - jeunes.loc["M", "total"]
print(f"Coût moyen à 26-35 ans : femmes CHF {chf(jeunes.loc['F', 'total'])}, hommes CHF {chf(jeunes.loc['M', 'total'])}")
print(f"Écart : CHF {chf(ecart)}, dont maternité : CHF {chf(jeunes.loc['F', 'maternite'])} "
      f"({jeunes.loc['F', 'maternite'] / ecart:.0%} de l'écart)")
jeunes.T

### 3.3 Les accidents

L'assurance obligatoire des soins ne couvre l'accident que pour les personnes qui ne sont pas assurées contre les accidents par leur employeur (LAA) : les salarié·es travaillant au moins 8 heures par semaine sont couvert·es par la LAA, et leurs accidents n'apparaissent pas dans ces données. La variable `couverture_accident` indique les personnes couvertes par la LAMal.

In [ ]:
accidents = couts_cause.groupby("classe_ofsp", observed=True).agg(couverture=("couverture_accident", "mean"),
                                                                   cout_accident=("accident", "mean"))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
accidents["couverture"].plot.bar(ax=axes[0], color="#2A78D6", width=0.75)
axe_pct(axes[0])
axes[0].set(title="Couvert·es contre les accidents par la LAMal", xlabel="Classe d'âge", ylabel="Part des assuré·es")
accidents["cout_accident"].plot.bar(ax=axes[1], color=COULEURS_CAUSE["accident"], width=0.75)
axe_chf(axes[1])
axes[1].set(title="Coût moyen des accidents par personne", xlabel="Classe d'âge", ylabel="CHF")
for ax in axes:
    ax.tick_params(axis="x", rotation=0)

**Questions**
1. À quels âges les dépenses des femmes dépassent-elles celles des hommes ? Quelle part de l'écart la maternité explique-t-elle à 26-35 ans, et quelles hypothèses avancer pour le reste ?
2. Les prestations de maternité ne donnent lieu à aucune participation (franchise, quote-part). Quelle est la justification de cette règle ?
3. Pourquoi les accidents apparaissent-ils surtout après 65 ans dans ces données ?
4. Pour comparer des personnes atteintes d'une maladie chronique aux autres assuré·es, faut-il exclure les prestations de cause « accident » ? Et celles de cause « maternité » ? Justifier.

**Réponses du groupe** *(double-cliquer sur la cellule pour la modifier)*

### 3.4 Concentration des dépenses

La courbe suivante classe les assuré·es du plus coûteux au moins coûteux et cumule leurs dépenses.

In [ ]:
tri = np.sort(couts_cause["total"].to_numpy())[::-1]
cumul = tri.cumsum() / tri.sum()
part_personnes = np.arange(1, len(tri) + 1) / len(tri)

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.plot(part_personnes, cumul, color="#2A78D6", lw=2)
ax.plot([0, 1], [0, 1], color="#7F8C8D", ls=":", lw=1)
for p in (0.01, 0.10, 0.20, 0.50):
    valeur = cumul[int(p * len(tri)) - 1]
    ax.plot(p, valeur, "o", ms=7, color="#2A78D6")
    ax.annotate(f"{p:.0%} des assuré·es : {valeur:.0%} des coûts", (p, valeur), xytext=(10, -14), textcoords="offset points")
axe_pct(ax)
axe_pct(ax, "x")
ax.set(title="Concentration des dépenses", xlabel="Assuré·es, du plus au moins coûteux", ylabel="Part cumulée des dépenses");

In [ ]:
concentration = pd.Series({
    "1 % les plus coûteux": cumul[int(0.01 * len(tri)) - 1],
    "5 % les plus coûteux": cumul[int(0.05 * len(tri)) - 1],
    "10 % les plus coûteux": cumul[int(0.10 * len(tri)) - 1],
    "20 % les plus coûteux": cumul[int(0.20 * len(tri)) - 1],
    "50 % les moins coûteux": 1 - cumul[len(tri) // 2 - 1],
}).map("{:.1%}".format).to_frame("Part des dépenses")
print(f"Assuré·es sans aucun décompte en 2024 : {(couts_cause['total'] <= 0).mean():.1%}")
concentration

In [ ]:
# Qui sont les 10 % les plus coûteux ?
seuil = couts_cause["total"].quantile(0.9)
hospitalises = contacts.loc[contacts["type_prestation"] == "stationnaire", "id_assure"].unique()
deces = assures_ok.loc[assures_ok["motif_fin_couverture"] == "deces", "id_assure"]
substances = med_ok.groupby("id_assure")["code_atc"].nunique()

profil_couts = couts_cause.assign(
    groupe=np.where(couts_cause["total"] >= seuil, "10 % les plus coûteux", "90 % restants"),
    hospitalise=couts_cause.index.isin(hospitalises),
    decede=couts_cause.index.isin(deces),
    substances=substances.reindex(couts_cause.index).fillna(0),
)
profil_couts.groupby("groupe").agg(
    personnes=("total", "size"),
    age_moyen=("age", "mean"),
    pct_femmes=("sexe", lambda s: 100 * (s == "F").mean()),
    pct_hospitalises=("hospitalise", lambda s: 100 * s.mean()),
    pct_decedes=("decede", lambda s: 100 * s.mean()),
    substances_differentes=("substances", "mean"),
    cout_moyen=("total", "mean"),
).T

**Questions**
1. Les dépenses suivent-elles la « règle des 80/20 » ? Quelle part des dépenses représentent les 1 % les plus coûteux, et la moitié la moins coûteuse des assuré·es ?
2. Pourquoi une part importante des assuré·es n'a-t-elle aucun décompte dans l'année ?
3. Qu'est-ce qui distingue les 10 % les plus coûteux ? Quelles conséquences ce degré de concentration a-t-il pour le financement de l'assurance obligatoire (solidarité, compensation des risques entre assureurs) ?

**Réponses du groupe** *(double-cliquer sur la cellule pour la modifier)*

---
## 4. Personnes traitées pour hypertension

Les sections précédentes ont montré que les dépenses augmentent avec l'âge et se concentrent sur une minorité de personnes, souvent atteintes de maladies chroniques. Cette section étudie l'une d'elles, l'hypertension artérielle. Conformément à la discussion de la section 3, le recours aux soins et les coûts sont calculés sur les prestations de cause « maladie » et sur les médicaments : les accidents et la maternité, dont l'observation dépend de la couverture et du sexe, en sont exclus.

### 4.1 Identification des personnes traitées

Les données de facturation ambulatoire ne comportent pas de diagnostic. Les personnes traitées sont donc identifiées à partir des médicaments délivrés, selon la définition opérationnelle suivante : **au moins deux délivrances en 2024 d'un antihypertenseur**, c'est-à-dire d'un médicament dont le code ATC commence par l'un des préfixes ci-dessous. Les personnes ainsi identifiées forment le groupe « HTA traitée » (hypertension artérielle traitée).

| C02 | C03 | C07 | C08 | C09 |
|---|---|---|---|---|
| action centrale / alpha-bloquants | diurétiques | bêtabloquants | inhibiteurs calciques | IEC / sartans (ARA) |

In [ ]:
# Paramètres de la définition opérationnelle (modifiables, voir les analyses complémentaires)
CLASSES_ANTIHYPERTENSEURS = ["C02", "C03", "C07", "C08", "C09"]
SEUIL_DELIVRANCES = 2

In [ ]:
antihta = med_ok[med_ok["code_atc"].str[:3].isin(CLASSES_ANTIHYPERTENSEURS)]
nb_delivrances = antihta.groupby("id_assure").size()
ids_hta = nb_delivrances[nb_delivrances >= SEUIL_DELIVRANCES].index

assures_ok["groupe"] = np.where(assures_ok["id_assure"].isin(ids_hta), "HTA traitée", "Autres assuré·es")
hta = assures_ok["groupe"] == "HTA traitée"

print(f"{hta.sum():,} personnes traitées pour hypertension, soit {hta.mean():.1%} des assuré·es")

In [ ]:
ax = nb_delivrances.clip(upper=20).value_counts().sort_index().plot.bar(color="#34495E", width=0.8)
ax.axvline(SEUIL_DELIVRANCES - 1.5, color="#C0392B", ls="--")
ax.set(title="Nombre de délivrances d'antihypertenseurs par personne (20 = 20 ou plus)",
       xlabel="Délivrances en 2024", ylabel="Nombre de personnes")
ax.tick_params(axis="x", rotation=0);

Les bêtabloquants et les diurétiques ont d'autres indications que l'hypertension. Le tableau suivant décrit les personnes identifiées sur la base d'une seule classe thérapeutique, bêtabloquant (C07) ou diurétique de l'anse (C03C).

In [ ]:
classes_par_personne = antihta.groupby("id_assure")["code_atc"].agg(lambda s: set(s.str[:3]))
une_seule_classe = classes_par_personne[classes_par_personne.apply(len) == 1].index.intersection(ids_hta)

a_verifier = (antihta[antihta["id_assure"].isin(une_seule_classe)
                      & antihta["code_atc"].str.startswith(("C07", "C03C"))]
              .drop_duplicates(["id_assure", "code_atc"])
              .merge(referentiel[["code_atc", "substance"]], on="code_atc")
              .merge(assures_ok[["id_assure", "age", "sexe"]], on="id_assure"))

anticoagules = med_ok.loc[med_ok["code_atc"].str.startswith("B01AF"), "id_assure"].unique()
a_verifier["aussi_anticoagulant"] = a_verifier["id_assure"].isin(anticoagules)

(a_verifier.groupby("substance")
 .agg(personnes=("id_assure", "nunique"), age_median=("age", "median"),
      pct_femmes=("sexe", lambda s: 100 * (s == "F").mean()),
      pct_avec_anticoagulant=("aussi_anticoagulant", lambda s: 100 * s.mean()))
 .sort_values("personnes", ascending=False))

**Questions**
1. La proportion de personnes identifiées est-elle plausible ?
2. Quelles personnes cette définition omet-elle, et lesquelles inclut-elle à tort ? On pourra considérer l'âge et le sexe des personnes traitées par propranolol, ainsi que l'association d'un bêtabloquant et d'un anticoagulant.
3. Cette proportion constitue-t-elle une estimation de la prévalence de l'hypertension ?

**Analyse complémentaire.** Modifier `SEUIL_DELIVRANCES` (par exemple 1 ou 4), puis exécuter les cellules suivantes (menu *Exécution*). Quelle est la sensibilité des résultats à la définition retenue ?

**Réponses du groupe** *(double-cliquer sur la cellule pour la modifier)*

### 4.2 Caractéristiques des personnes traitées

Les autres maladies chroniques sont, elles aussi, approchées par les médicaments délivrés : antidiabétiques (A10), statines (C10) et anticoagulants oraux directs (B01AF).

In [ ]:
prevalence = assures_ok.groupby(["classe_age", "sexe"], observed=True)["groupe"].apply(lambda g: (g == "HTA traitée").mean())
ax = prevalence.reset_index(name="part").pipe(
    lambda d: sns.barplot(data=d, x="classe_age", y="part", hue="sexe", palette=COULEURS_SEXE, hue_order=["F", "M"]))
axe_pct(ax)
ax.set(title="Part des personnes traitées pour hypertension", xlabel="Classe d'âge", ylabel="");

In [ ]:
# Indicateurs de comorbidité fondés sur les délivrances
PROXYS = {"Diabète (A10)": "A10", "Statine (C10)": "C10", "Anticoagulant (B01AF)": "B01AF"}
for nom, prefixe in PROXYS.items():
    assures_ok[nom] = assures_ok["id_assure"].isin(med_ok.loc[med_ok["code_atc"].str.startswith(prefixe), "id_assure"])

profil = assures_ok.groupby("groupe").agg(
    personnes=("id_assure", "size"),
    age_moyen=("age", "mean"),
    pct_femmes=("sexe", lambda s: 100 * (s == "F").mean()),
    pct_subside=("subside", lambda s: 100 * s.mean()),
    **{f"% {nom}": (nom, lambda s: 100 * s.mean()) for nom in PROXYS},
).T[GROUPES]
profil

In [ ]:
comorbidites = assures_ok.groupby("groupe")[list(PROXYS)].mean().T[GROUPES]
ax = comorbidites.plot.barh(color=COULEURS, width=0.75)
ax.invert_yaxis()
axe_pct(ax, "x")
ax.set(title="Autres traitements chroniques", xlabel="Part des personnes", ylabel="");

La cellule suivante compare la distribution d'une variable entre les deux groupes. Paramètre modifiable : `VARIABLE` (`franchise_chf`, `modele_assurance`, `canton` ou `subside`).

In [ ]:
VARIABLE = "franchise_chf"   # paramètre modifiable

donnees = assures_ok.assign(franchise_chf=assures_ok["franchise_chf"].map(GROUPES_FRANCHISE))
repartition = pd.crosstab(donnees["groupe"], donnees[VARIABLE], normalize="index").loc[GROUPES]
if VARIABLE == "franchise_chf":
    repartition, couleurs = repartition[["300", "500", "1000-2000", "2500"]], RAMPE
else:
    couleurs = CATEGORIELLES[:repartition.shape[1]]
ax = repartition.plot.barh(stacked=True, color=couleurs, figsize=(8, 2.8), width=0.7, edgecolor="white", linewidth=1)
ax.invert_yaxis()
axe_pct(ax, "x")
ax.legend(title=VARIABLE, bbox_to_anchor=(1, 1), loc="upper left")
ax.set(title=f"Répartition selon « {VARIABLE} »", xlabel="", ylabel="");

**Questions**
1. Comment caractériser, en deux phrases, les personnes traitées pour hypertension ?
2. Comment expliquer la fréquence plus élevée de la franchise minimale dans ce groupe ?

**Réponses du groupe** *(double-cliquer sur la cellule pour la modifier)*

### 4.3 Traitements antihypertenseurs

Certains médicaments sont des associations fixes, qui réunissent plusieurs classes thérapeutiques sous un seul code ATC (par exemple `C09DA` : antagoniste de l'angiotensine II et diurétique). Chaque code est donc converti en la ou les classes qu'il contient.

In [ ]:
CLASSES = {   # début du code ATC → classes contenues
    "C09AA": ["IEC"], "C09BA": ["IEC", "Diurétique"], "C09BB": ["IEC", "Inhibiteur calcique"],
    "C09CA": ["Sartan (ARA)"], "C09DA": ["Sartan (ARA)", "Diurétique"],
    "C09DB": ["Sartan (ARA)", "Inhibiteur calcique"],
    "C09DX": ["Sartan (ARA)", "Inhibiteur calcique", "Diurétique"],
    "C08": ["Inhibiteur calcique"], "C07": ["Bêtabloquant"], "C02": ["Autre antihypertenseur"],
    "C03A": ["Diurétique"], "C03B": ["Diurétique"], "C03C": ["Diurétique de l'anse"], "C03D": ["Antagoniste aldostérone"],
}


def classes_du_code(code_atc):
    for debut in sorted(CLASSES, key=len, reverse=True):
        if code_atc.startswith(debut):
            return CLASSES[debut]
    return ["Non classé"]


trt = antihta[antihta["id_assure"].isin(ids_hta)].copy()
trt["classe"] = trt["code_atc"].apply(classes_du_code)
trt["combinaison_fixe"] = trt["classe"].apply(len) > 1
trt_classes = trt.explode("classe")

par_personne = pd.DataFrame({
    "nb_codes": trt.groupby("id_assure")["code_atc"].nunique(),
    "nb_classes": trt_classes.groupby("id_assure")["classe"].nunique(),
    "combinaison_fixe": trt.groupby("id_assure")["combinaison_fixe"].any(),
})
print(f"{par_personne['combinaison_fixe'].mean():.0%} des personnes reçoivent au moins une combinaison fixe")

In [ ]:
part_classes = trt_classes.groupby("classe")["id_assure"].nunique().div(len(par_personne)).sort_values()
ax = part_classes.plot.barh(color=COULEURS["HTA traitée"])
axe_pct(ax, "x")
ax.set(title="Classes utilisées (au moins une fois en 2024)", xlabel="Part des personnes traitées", ylabel="");

In [ ]:
comparaison = pd.DataFrame({
    "Compté en codes ATC": par_personne["nb_codes"].clip(upper=4).value_counts(normalize=True),
    "Compté en classes": par_personne["nb_classes"].clip(upper=4).value_counts(normalize=True),
}).sort_index().rename(index={4: "4+"})
ax = comparaison.plot.bar(color=["#BDC3C7", COULEURS["HTA traitée"]], width=0.75)
axe_pct(ax)
ax.tick_params(axis="x", rotation=0)
ax.set(title="Nombre de médicaments antihypertenseurs par personne", xlabel="Nombre", ylabel="Part des personnes");

In [ ]:
par_personne = par_personne.join(assures_ok.set_index("id_assure")["classe_age"])
polytherapie = pd.crosstab(par_personne["classe_age"], par_personne["nb_classes"].clip(upper=3), normalize="index")
ax = polytherapie.rename(columns={1: "1 classe", 2: "2 classes", 3: "3 classes ou +"}).plot.bar(
    stacked=True, color=["#F5B7B1", "#E74C3C", "#922B21"], width=0.75, edgecolor="white", linewidth=1)
axe_pct(ax)
ax.tick_params(axis="x", rotation=0)
ax.legend(bbox_to_anchor=(1, 1), loc="upper left")
ax.set(title="Nombre de classes selon l'âge", xlabel="Classe d'âge", ylabel="");

**Questions**
1. Quelles sont les classes les plus utilisées ? La monothérapie est-elle majoritaire, et comment évolue-t-elle avec l'âge ?
2. Pourquoi un dénombrement des codes ATC sous-estime-t-il la fréquence des associations thérapeutiques ?

**Réponses du groupe** *(double-cliquer sur la cellule pour la modifier)*

### 4.4 Recours aux soins

Le recours aux soins est mesuré par le nombre de contacts de cause « maladie » (définis à la section 1) par personne et par type de prestation.

In [ ]:
contacts_maladie = contacts[contacts["cause"] == "maladie"]
recours = (pd.crosstab(contacts_maladie["id_assure"], contacts_maladie["type_prestation"])
           .reindex(assures_ok["id_assure"], fill_value=0))
recours["hospitalise"] = recours["stationnaire"] > 0
recours = recours.join(assures_ok.set_index("id_assure")[["groupe", "classe_age"]])

moyennes = recours.groupby("groupe")[["medecin_generaliste", "medecin_specialiste", "laboratoire",
                                      "ambulatoire_hospitalier", "physiotherapie"]].mean().T[GROUPES]
ax = moyennes.plot.barh(color=COULEURS, width=0.75)
ax.invert_yaxis()
ax.set(title="Nombre moyen de contacts par personne en 2024", xlabel="Contacts", ylabel="");

In [ ]:
print("Part des personnes hospitalisées au moins une fois :")
recours.groupby("groupe")["hospitalise"].mean().loc[GROUPES].map("{:.1%}".format).to_frame("")

Les personnes traitées étant en moyenne plus âgées, la comparaison est répétée par classe d'âge afin de distinguer l'effet de l'âge de celui de la condition étudiée.

In [ ]:
ax = sns.pointplot(data=recours, x="classe_age", y="medecin_generaliste", hue="groupe",
                   palette=COULEURS, hue_order=GROUPES, errorbar=None)
ax.set(title="Consultations chez le généraliste, à âge égal", xlabel="Classe d'âge", ylabel="Contacts moyens par personne")
ax.legend(title="");

In [ ]:
specialites = (contacts_maladie[contacts_maladie["type_prestation"] == "medecin_specialiste"]
               .merge(assures_ok[["id_assure", "groupe"]], on="id_assure"))
repartition = pd.crosstab(specialites["specialite"], specialites["groupe"], normalize="columns")[GROUPES]
ax = repartition.sort_values("HTA traitée").plot.barh(color=COULEURS, width=0.75, figsize=(8, 5))
axe_pct(ax, "x")
ax.set(title="Répartition des consultations de spécialistes", xlabel="Part des consultations", ylabel="");

**Questions**
1. Les personnes traitées recourent-elles davantage aux soins, et pour quels types de prestations ?
2. La différence persiste-t-elle à âge égal ? Quels autres facteurs pourraient l'expliquer ?

**Réponses du groupe** *(double-cliquer sur la cellule pour la modifier)*

**Analyse complémentaire : continuité des soins.** La variable `id_prestataire` permet de savoir si une personne consulte toujours le même généraliste.

In [ ]:
generalistes = prest_ok[(prest_ok["type_prestation"] == "medecin_generaliste") & (prest_ok["cause"] == "maladie")
                        & (prest_ok["cout_chf"] > 0)]
continuite = generalistes.groupby("id_assure").agg(jours=("date_prestation", "nunique"),
                                                   medecins=("id_prestataire", "nunique"))
continuite = continuite[continuite["jours"] >= 3].join(assures_ok.set_index("id_assure")["groupe"])
print("Personnes avec au moins 3 consultations de généraliste n'ayant vu qu'un seul généraliste :")
continuite.groupby("groupe")["medecins"].apply(lambda s: (s == 1).mean()).loc[GROUPES].map("{:.0%}".format).to_frame("")

**Question.** Pourquoi la continuité des soins est-elle un enjeu particulier pour les personnes atteintes d'une maladie chronique ? Quel lien avec les modèles d'assurance « médecin de famille » ?

### 4.5 Coûts

Le coût comprend les prestations de cause « maladie » et les médicaments, décomptés en 2024 à la charge de l'assurance obligatoire des soins.

In [ ]:
prest_maladie = prest_ok[prest_ok["cause"] == "maladie"]
couts = prest_maladie.pivot_table(index="id_assure", columns="type_prestation", values="cout_chf", aggfunc="sum")
couts["med_antihypertenseurs"] = antihta.groupby("id_assure")["cout_chf"].sum()
couts["med_autres"] = med_ok[~med_ok.index.isin(antihta.index)].groupby("id_assure")["cout_chf"].sum()
couts = couts.reindex(assures_ok["id_assure"]).fillna(0)

POSTES = list(couts.columns)
couts["total"] = couts[POSTES].sum(axis=1)
couts = couts.join(assures_ok.set_index("id_assure")[["groupe", "classe_age", "franchise_chf", "motif_fin_couverture"]])

(couts.groupby("groupe")["total"]
 .describe(percentiles=[0.5, 0.9])[["count", "mean", "50%", "90%", "max"]]
 .rename(columns={"count": "personnes", "mean": "moyenne", "50%": "médiane", "90%": "90e centile"})
 .loc[GROUPES].round(0))

In [ ]:
ax = sns.histplot(data=couts[couts["total"] > 0], x="total", hue="groupe", hue_order=GROUPES,
                  palette=COULEURS, log_scale=True, bins=40, stat="density", common_norm=False, element="step")
ax.set(title="Distribution du coût annuel par personne (échelle logarithmique)", xlabel="CHF", ylabel="");

In [ ]:
NOMS = {"stationnaire": "Hospitalisation", "ambulatoire_hospitalier": "Ambulatoire hospitalier",
        "medecin_generaliste": "Généraliste", "medecin_specialiste": "Spécialiste", "laboratoire": "Laboratoire",
        "physiotherapie": "Physiothérapie", "soins_domicile": "Soins à domicile",
        "med_antihypertenseurs": "Médicaments : antihypertenseurs", "med_autres": "Médicaments : autres"}
postes = couts.groupby("groupe")[POSTES].mean().loc[GROUPES]
postes = postes[postes.sum().sort_values(ascending=False).index]
ax = postes.rename(columns=NOMS).plot.bar(stacked=True, color=CATEGORIELLES[:len(POSTES)], width=0.6, figsize=(7, 5),
                                          edgecolor="white", linewidth=1)
axe_chf(ax)
ax.tick_params(axis="x", rotation=0)
ax.legend(bbox_to_anchor=(1, 1), loc="upper left")
ax.set(title="Coût moyen par personne et par poste", xlabel="", ylabel="CHF");

print(f"Antihypertenseurs = {postes.loc['HTA traitée', 'med_antihypertenseurs'] / postes.loc['HTA traitée'].sum():.1%} "
      "du coût moyen des personnes traitées")

In [ ]:
ax = sns.pointplot(data=couts, x="classe_age", y="total", hue="groupe", palette=COULEURS,
                   hue_order=GROUPES, estimator="mean", errorbar=None)
axe_chf(ax)
ax.set(title="Coût moyen par personne, à âge égal", xlabel="Classe d'âge", ylabel="CHF")
ax.legend(title="");

In [ ]:
poids = couts.groupby("groupe").agg(personnes=("total", "size"), couts=("total", "sum"))
poids = (poids / poids.sum()).loc[GROUPES]
ax = poids.T.rename(index={"personnes": "Part des assuré·es", "couts": "Part des coûts"}).plot.barh(
    stacked=True, color=COULEURS, width=0.6, figsize=(8, 2.5), edgecolor="white", linewidth=1)
ax.invert_yaxis()
axe_pct(ax, "x")
ax.legend(bbox_to_anchor=(1, 1), loc="upper left")
ax.set(title="Part des personnes traitées dans l'effectif et dans les coûts", xlabel="", ylabel="");

plus_couteux = couts.sort_values("total", ascending=False).iloc[:len(couts) // 10]
part_hta_top10 = (plus_couteux["groupe"] == "HTA traitée").mean()
print(f"Personnes traitées pour HTA parmi les 10 % les plus coûteux : {part_hta_top10:.0%} "
      f"(contre {hta.mean():.0%} de l'ensemble des assuré·es)")

**Questions**
1. Compte tenu de la distribution des coûts, quel indicateur (moyenne ou médiane) convient-il de présenter à un·e décideur·e, et dans quel but ?
2. Quelle part des coûts des personnes traitées est attribuable aux antihypertenseurs ?
3. L'écart de coût persiste-t-il à âge égal ? Peut-on l'attribuer à l'hypertension ?

**Réponses du groupe** *(double-cliquer sur la cellule pour la modifier)*

### Analyses complémentaires

In [ ]:
# Coût selon la franchise
ax = sns.barplot(data=couts, x="franchise_chf", y="total", hue="groupe", palette=COULEURS,
                 hue_order=GROUPES, estimator="mean", errorbar=None)
axe_chf(ax)
ax.set(title="Coût moyen selon la franchise", xlabel="Franchise (CHF)", ylabel="CHF")
ax.legend(title="");

**Question.** Les coûts observés des personnes ayant choisi une franchise de 2 500 CHF sont très faibles. Quels mécanismes peuvent l'expliquer, et lequel relève d'un artefact de mesure ?

In [ ]:
# Personnes non assurées toute l'année
couts["motif"] = couts["motif_fin_couverture"].fillna("assuré·e toute l'année")
couts.groupby("motif")["total"].agg(["size", "mean", "median"]).round(0)

**Question.** Que suggèrent ces résultats concernant les coûts en fin de vie ? Quel biais introduit la comparaison de personnes assurées pendant des durées différentes ?

**Réponses du groupe** *(double-cliquer sur la cellule pour la modifier)*

---
## 5. Synthèse et rédaction de la note

Le tableau suivant rassemble les principaux résultats. Les valeurs dépendent des choix effectués dans les sections précédentes.

In [ ]:
# Synthèse des principaux résultats
g = couts.groupby("groupe")["total"]
chiffres = pd.Series({
    "Personnes traitées pour hypertension": f"{hta.sum():,} ({hta.mean():.1%} des assuré·es)",
    "Âge moyen (traitées / autres)": f"{profil.loc['age_moyen', 'HTA traitée']:.0f} / {profil.loc['age_moyen', 'Autres assuré·es']:.0f} ans",
    "Part de femmes parmi les traitées": f"{profil.loc['pct_femmes', 'HTA traitée']:.0f}%",
    "Traitées avec ≥ 2 classes": f"{(par_personne['nb_classes'] >= 2).mean():.0%}",
    "Généraliste, contacts/an (traitées / autres)":
        f"{moyennes.loc['medecin_generaliste', 'HTA traitée']:.1f} / {moyennes.loc['medecin_generaliste', 'Autres assuré·es']:.1f} par an",
    "Hospitalisées ≥ 1 fois (traitées / autres)":
        f"{recours.groupby('groupe')['hospitalise'].mean()['HTA traitée']:.0%} / {recours.groupby('groupe')['hospitalise'].mean()['Autres assuré·es']:.0%}",
    "Coût moyen (traitées / autres)": f"{g.mean()['HTA traitée']:,.0f} / {g.mean()['Autres assuré·es']:,.0f} CHF",
    "Coût médian (traitées / autres)": f"{g.median()['HTA traitée']:,.0f} / {g.median()['Autres assuré·es']:,.0f} CHF",
    "Part des coûts totaux portée par les traitées": f"{poids.loc['HTA traitée', 'couts']:.0%}",
    "Part des traitées parmi les 10 % les plus coûteux": f"{part_hta_top10:.0%}",
    "Part des antihypertenseurs dans leur coût": f"{postes.loc['HTA traitée', 'med_antihypertenseurs'] / postes.loc['HTA traitée'].sum():.1%}",
}).str.replace(",", "'")
chiffres.to_frame("Valeur")

### Structure de la note

**Destinataire :** responsable de la santé publique cantonale.
**Objet :** personnes traitées pour hypertension artérielle : profil, recours aux soins et coûts.

1. **Population concernée.** Effectif, proportion et principales caractéristiques.
2. **Traitements et recours aux soins.** Éléments distinguant les personnes traitées des autres assuré·es.
3. **Coûts associés.** Ordres de grandeur, principaux déterminants et place dans la concentration des dépenses.
4. **Messages principaux.** Deux ou trois, formulés en vue de la décision.
5. **Limite d'interprétation.** La principale limite et ses implications pour la décision.

La note tient sur une page et comporte au plus un graphique. Les chiffres sont arrondis et le vocabulaire technique propre à l'analyse des données est évité.